# Clase 2 · Árboles binomiales monoperiodo y cobertura delta

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prof-rodolfo-medina/modelizacion-derivados-carteras/blob/main/sesiones/semana-02/clase-02-arboles-binomiales-monoperiodo.ipynb)

**Semana 2 (09–13/11/2026) · Tema 3 · 60 min + presentación de la Actividad 1**

Objetivos:
1. Calcular la prima racional de una opción europea en un árbol de un periodo (Métodos I y II).
2. Mostrar con números por qué cualquier otro precio genera arbitraje.
3. Implementar la cobertura delta de un corredor que vende calls.

In [1]:
# Configuración: funciona en local (repositorio clonado) y en Google Colab
import sys, subprocess, pathlib
try:
    import mvdc
except ImportError:
    if "google.colab" in sys.modules:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/prof-rodolfo-medina/modelizacion-derivados-carteras.git"], check=True)
    else:
        for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (base / "src" / "mvdc").exists():
                sys.path.insert(0, str(base / "src"))
                break
    import mvdc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print("mvdc", mvdc.__version__)

mvdc 2026.11


## 1. Método I — cartera de acciones y opciones sin riesgo

Con $b^*=-1$ (vendemos una opción) la cartera $a^*S - V$ es sin riesgo si

$$a^* = \frac{U-D}{S_u-S_d}=\Delta, \qquad V_0 = a^*S_0 + (U-a^*S_u)e^{-r\tau}.$$

## 2. Método II — réplica con acciones y bonos

$$V_0 = e^{-r\tau}\left(q^*U + (1-q^*)D\right), \qquad q^*=\frac{e^{r\tau}S_0-S_d}{S_u-S_d}.$$

In [2]:
from mvdc import binomial as b

res = b.one_period_price(S0=100, Su=120, Sd=90, K=105, r=0.04879, tau=1, kind="call")
pd.Series(res).round(4)

U                  15.0000
D                   0.0000
delta               0.5000
bonos             -42.8571
q                   0.5000
prima_metodo_I      7.1429
prima_metodo_II     7.1429
dtype: float64

## 3. ¿Qué pasa si el mercado cotiza otra prima? — Arbitraje

Reproducimos la discusión del tema con $V=7.25$ (cara) y $V=7.00$ (barata).

In [3]:
from math import exp

S0, Su, Sd, K, r = 100, 120, 90, 105, 0.04879

def arbitraje(V_mercado):
    V_justo = b.one_period_price(S0, Su, Sd, K, r)["prima_metodo_I"]
    signo = -1 if V_mercado > V_justo else +1        # vendemos si está cara, compramos si está barata
    a, bopc = -signo * 1, signo * 2                   # b = -2a  (una acción, dos opciones)
    coste0 = a * S0 + bopc * V_mercado               # > 0: pedimos prestado; < 0: invertimos
    finales = {}
    for nombre, S in [("sube", Su), ("baja", Sd)]:
        finales[nombre] = a * S + bopc * max(S - K, 0) - coste0 * exp(r)
    return V_justo, finales

for V in [7.25, 7.00]:
    Vj, g = arbitraje(V)
    print(f"V mercado = {V:.2f} (justo {Vj:.4f}) → ganancia sin riesgo: sube {g['sube']:.4f}, baja {g['baja']:.4f}")

V mercado = 7.25 (justo 7.1429) → ganancia sin riesgo: sube 0.2250, baja 0.2250
V mercado = 7.00 (justo 7.1429) → ganancia sin riesgo: sube 0.3000, baja 0.3000


## 4. Cobertura delta del corredor (Sección 3.3)

El corredor vende $N=100\,000$ calls ($S_0=60$, $S_u=80$, $S_d=50$, $K=65$, $r=0.048$) y compra
$a^*N$ acciones financiadas en el banco.

In [4]:
for prima in [6.00, b.one_period_price(60, 80, 50, 65, 0.048)["prima_metodo_I"], 6.25]:
    r_ = b.delta_hedge_pnl(60, 80, 50, 65, 0.048, 1, prima, 100_000)
    print(f"prima cobrada {prima:7.4f} → ganancia si sube {r_['ganancia_sube']:>12,.2f} | si baja {r_['ganancia_baja']:>12,.2f}")

prima cobrada  6.0000 → ganancia si sube   -18,009.57 | si baja   -18,009.57
prima cobrada  6.1717 → ganancia si sube        -0.00 | si baja        -0.00
prima cobrada  6.2500 → ganancia si sube     8,219.69 | si baja     8,219.69


> **Nota:** en los apuntes la ganancia con prima 6.25 aparece como 6 250, porque se redondea
$e^{0.048}\approx 1.05$. Con el factor exacto la ganancia es ≈ 8 219.69 en ambos escenarios —
lo importante es que **coincide** en los dos.

## 5. Ejercicio en clase

Repite el análisis para una **put** europea con los mismos datos ($K=65$).
¿Qué signo tiene la delta? ¿Qué hace ahora el corredor para cubrirse?

In [5]:
# Tu código aquí
# b.one_period_price(60, 80, 50, 65, 0.048, kind="put")

## 6. Presentación de la Actividad 1 — *Long straddle*

El enunciado completo, la rúbrica, la plantilla y la lista de comprobación están en
[`actividades/actividad-1-straddle/`](../../actividades/actividad-1-straddle/README.md).

**Entrega: lunes 23/11/2026, 23:59** (por el aula virtual de UNIR, no por GitHub).

Ideas a repasar hoy: B/P de call larga y put larga (Clase 1), puntos de equilibrio,
por qué la estrategia *apuesta por volatilidad* y no por dirección.